# Optional court model fine-tuning
The local app already uses the original pretrained TennisCourtDetector. Train this model only when you want to test a court-specific improvement. This keeps the original 15-channel heatmap architecture (14 court keypoints plus a centre point), not a human-pose checkpoint. Downloaded upstream code is not bundled with the local app.

## 1. Setup
Select **Runtime → Change runtime type → GPU**. This installs the versioned project wheel from the public GitHub release, including the same model and export code used locally.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import subprocess, sys
WHEEL = 'https://github.com/Paulyang5049/tennis-ai-local/releases/download/v0.1.0/tennis_ai_local-0.1.0-py3-none-any.whl'
subprocess.run([sys.executable, '-m', 'pip', 'install', WHEEL], check=True)
print('If pip replaced an already imported torch/numpy, restart the session before continuing.')

In [ ]:
import torch, importlib.metadata
assert torch.cuda.is_available(), 'Choose a GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))
print({n: importlib.metadata.version(n) for n in ['torch', 'ultralytics', 'numpy']})
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/TennisAI')
DRIVE.mkdir(parents=True, exist_ok=True)
print('Checkpoints will be saved to', DRIVE)

## 2. Download and inspect
The original dataset contains 8,841 images according to its README. This step validates annotations, removes file duplicates and groups filename families before splitting. If the download or grouping fails, keep the original local court model and resolve the data issue. The dataset download can take several minutes.

In [ ]:
from tennis_ai.training import prepare_court
WORK = Path('/content/court-work')
data, audit = prepare_court(WORK)
print(audit)

In [ ]:
import cv2, json, matplotlib.pyplot as plt
sample = json.loads((data/'data_train.json').read_text())[0]
image = cv2.imread(str(data/'images'/(sample['id']+'.png')))
for i, (x,y) in enumerate(sample['kps']):
    cv2.circle(image, (int(x),int(y)), 5, (0,255,0), -1)
    cv2.putText(image, str(i), (int(x),int(y)), cv2.FONT_HERSHEY_SIMPLEX, .5, (0,0,255), 1)
plt.figure(figsize=(14,8)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis('off');

## 3. Verify pretrained inference and training
The helper loads the original weights, verifies finite 15×360×640 outputs, then runs two training and validation batches.

In [ ]:
from tennis_ai.training import train_court
train_court(WORK, DRIVE/'court', smoke=True)

## 4. Optional fine-tuning
Change `RUN_FINE_TUNING` to True to run up to 50 epochs. Adam, learning rate 1e-5, batch size 2. Drive checkpoints include optimizer state and completed epoch; a rerun resumes automatically. Selection uses validation heatmap loss. This does not establish real-world court accuracy.

In [ ]:
RUN_FINE_TUNING = False
if RUN_FINE_TUNING:
    bundle = train_court(WORK, DRIVE/'court', epochs=50)
    print('Bundle:', bundle)
else:
    print('Skipped: retain the pretrained court model.')

## 5. Export
After fine-tuning, download the bundle, unzip inside `models/`, and verify in the local app. Compare court-keypoint error on held-out phone and broadcast annotations with `tennis-ai evaluate`. Keep the original model if the candidate is worse.

In [ ]:
import shutil
if RUN_FINE_TUNING:
    archive = shutil.make_archive('/content/court-bundle', 'zip', bundle)
    files.download(archive)